#Initial

###Imports

In [ ]:
# ===========================================
# Install dependencies
# ===========================================
!pip install -U transformers accelerate datasets sentencepiece -q



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.5/511.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 17.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.


In [ ]:
import transformers
print(transformers.__version__)


4.57.1


###Main1

In [ ]:
# ===========================================
# Imports
# ===========================================
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
import torch
import re
import json

# ===========================================
# Preprocess dataset (replace tags)
# ===========================================
# Define all tag types
tag_list = [
    "ACCOUNT_NUMBER", "ADDRESS", "APPLICATION_ID", "BRANCH_CODE", "CARD_NUMBER",
    "CHEQUE_NUMBER", "CUSTOMER_ID", "DATE_RANGE", "DOB", "DOCUMENT_ID", "EMAIL",
    "EMPLOYEE_ID", "FUND_ID", "ID_NUMBER", "IFSC_CODE", "NAME", "ORGANIZATION",
    "PHONE", "POLICY_ID", "SWIFT_CODE", "USERNAME", "USER_ID", "VEHICLE_ID"
]

# Function to replace tags with <SENSITIVE>
# def convert_to_simplified_tags(line):
#     for tag in tag_list:
#         line = re.sub(rf"<{tag}>", "<SENSITIVE>", line)
#         line = re.sub(rf"</{tag}>", "</SENSITIVE>", line)
#     return line

# # Read and rewrite JSONL dataset
# input_path = "new_cleaned.jsonl"
output_path = "new_cleaned.jsonl"

# with open(input_path, "r", encoding="utf-8") as infile, open(output_path, "w", encoding="utf-8") as outfile:
#     for line in infile:
#         data = json.loads(line)
#         data["original_query"] = convert_to_simplified_tags(data["original_query"])
#         data["sanitized_query"] = convert_to_simplified_tags(data["sanitized_query"])
#         outfile.write(json.dumps(data) + "\n")

# print(f"✅ Simplified dataset saved to {output_path}")

# ===========================================
# Load dataset
# ===========================================
dataset = load_dataset("json", data_files=output_path)
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]

# ===========================================
# Model & Tokenizer
# ===========================================
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Add only the simplified tags
# special_tokens = ["<SENSITIVE>", "</SENSITIVE>"]
# tokenizer.add_tokens(special_tokens)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

# ===========================================
# Preprocessing
# ===========================================
def preprocess(examples):
    inputs, targets = [], []
    for orig, sanitized in zip(examples["original_query"], examples["sanitized_query"]):
        if orig and sanitized:
            inputs.append("sanitize: " + str(orig))
            targets.append(str(sanitized))
    model_inputs = tokenizer(inputs, max_length=256, truncation=True)
    labels = tokenizer(targets, max_length=256, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

# ===========================================
# Data Collator
# ===========================================
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ===========================================
# Training Arguments (Fixed)
# ===========================================
args = TrainingArguments(
    output_dir="./sanitizer_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50
)

# ===========================================
# Trainer
# ===========================================
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# ===========================================
# Train the model
# ===========================================
trainer.train()

# ===========================================
# Save final model
# ===========================================
model.save_pretrained("./final_sanitizer_model")
tokenizer.save_pretrained("./final_sanitizer_model")

print("✅ Model saved to ./final_sanitizer_model")


Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saransh-naik14 (saransh-naik14-iiit-vadodara) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss
1,0.145400,0.057049
2,0.074600,0.031829
3,0.053500,0.022631
4,0.038500,0.019554
5,0.038900,0.018056
6,0.031500,0.017177
7,0.028200,0.016462
8,0.029900,0.016051
9,0.026700,0.015852
10,0.029400,0.015906


✅ Model saved to ./final_sanitizer_model


###TestNewb

In [ ]:
import re
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Example data
ground_truth = [
    "My name is <NAME>John Doe</NAME> and my phone is <PHONE>1234567890</PHONE>.",
    "Transfer from <ACCOUNT_NUMBER>778899</ACCOUNT_NUMBER> to <ACCOUNT_NUMBER>998877</ACCOUNT_NUMBER>."
]

predictions = [
    "My name is <NAME>John Doe</NAME> and my phone is 1234567890.",  # missed PHONE
    "Transfer from <ACCOUNT_NUMBER>778899</ACCOUNT_NUMBER> to 998877."  # missed second ACCOUNT_NUMBER
]

def extract_tags(text):
    """Extract tag names and values."""
    return re.findall(r"<([A-Z_]+)>(.*?)</\1>", text)

# Flattened lists for evaluation
true_tags, pred_tags = [], []

for gt_text, pred_text in zip(ground_truth, predictions):
    gt_pairs = extract_tags(gt_text)
    pred_pairs = extract_tags(pred_text)

    all_unique_tags = set([tag for tag, _ in gt_pairs] + [tag for tag, _ in pred_pairs])

    for tag in all_unique_tags:
        gt_has = any(t == tag for t, _ in gt_pairs)
        pred_has = any(t == tag for t, _ in pred_pairs)
        true_tags.append(gt_has)
        pred_tags.append(pred_has)

# Compute metrics
precision = precision_score(true_tags, pred_tags)
recall = recall_score(true_tags, pred_tags)
f1 = f1_score(true_tags, pred_tags)
accuracy = accuracy_score(true_tags, pred_tags)

print("🔍 Sanitization Model Evaluation:")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print(f"Accuracy:  {accuracy:.3f}")


🔍 Sanitization Model Evaluation:
Precision: 1.000
Recall:    0.667
F1-score:  0.800
Accuracy:  0.667


###Fixup

In [ ]:
def fix_tag_pairs(text):
    stack = []
    result = []
    tokens = re.split(r'(<SENSITIVE>|</SENSITIVE>)', text)
    for t in tokens:
        if t == '<SENSITIVE>':
            if stack:  # already open, close first
                result.append('</SENSITIVE>')
                stack.pop()
            stack.append('<SENSITIVE>')
            result.append(t)
        elif t == '</SENSITIVE>':
            if stack:
                stack.pop()
                result.append(t)
        else:
            result.append(t)
    if stack:
        result.append('</SENSITIVE>')  # close any leftover
    return ''.join(result)

import re

import re

def fix_sensitive_tags(text):
    # Step 1: Normalize whitespace and accidental casing
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'</?sensitive>', lambda m: m.group(0).upper(), text)

    # Step 2: Replace all `</SENSITIVE>` that appear before any `<SENSITIVE>` with `<SENSITIVE>`
    segments = []
    open_count = 0
    tokens = re.split(r'(<SENSITIVE>|</SENSITIVE>)', text)

    for tok in tokens:
        if tok == '</SENSITIVE>':
            if open_count == 0:
                # closing tag with no opener → treat as an opener
                segments.append('<SENSITIVE>')
                open_count += 1
            else:
                segments.append(tok)
                open_count -= 1
        elif tok == '<SENSITIVE>':
            segments.append(tok)
            open_count += 1
        else:
            segments.append(tok)

    # Step 3: If there's any unclosed tag, close it
    while open_count > 0:
        segments.append('</SENSITIVE>')
        open_count -= 1

    # Step 4: Join and clean double tags
    out = ''.join(segments)
    out = re.sub(r'(<SENSITIVE>\s*)+', '<SENSITIVE>', out)
    out = re.sub(r'(</SENSITIVE>\s*)+', '</SENSITIVE>', out)

    return out.strip()

bad_output = "My email is</SENSITIVE> saransh@gmail.com</SENSITIVE> and phone is</SENSITIVE> 9998877665</SENSITIVE>."
print(fix_sensitive_tags(bad_output))


# clean_output = fix_tag_pairs(model_output)


My email is<SENSITIVE>saransh@gmail.com</SENSITIVE>and phone is<SENSITIVE>9998877665</SENSITIVE>.


In [ ]:
# custom_query = "Phone number 9876543210, what’s my account balance?"
# print("\nOriginal Query:", custom_query)
# print("Sanitized Query:", sanitize_text(custom_query))
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_path = "./final_sanitizer_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

def sanitize_text(text):
    inputs = tokenizer("sanitize: " + text, return_tensors="pt", truncation=True)
    #inputs.append("Mark all sensitive info with tags: " + orig)
    outputs = model.generate(**inputs, max_length=256)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)



print(sanitize_text("I am so angry! I've been on hold for 20 minutes. My name is"
" Priya Sharma, my phone is 9876543210, and my account is 887766. I just want to "
"know my balance."))# and phone is 9998877665"))


I am so angry! I've been on hold for 20 minutes. My name is NAME>Priya Sharma/NAME>, my phone is PHONE>99776543210/PHONE>, and my account is ACCOUNT_NUMBER>887766/ACCOUNT_NUMBER>. I just want to know my balance.


In [ ]:
import re

def add_missing_opening_brackets(text):
    """
    Fix tags like NAME> or /NAME> by adding missing '<'.
    Works for all your known tag types.
    """
    tag_names = [
        "ACCOUNT_NUMBER", "ADDRESS", "APPLICATION_ID", "BRANCH_CODE",
        "CARD_NUMBER", "CHEQUE_NUMBER", "CUSTOMER_ID", "DATE_RANGE",
        "DOB", "DOCUMENT_ID", "EMAIL", "EMPLOYEE_ID", "FUND_ID",
        "ID_NUMBER", "IFSC_CODE", "NAME", "ORGANIZATION", "PHONE",
        "POLICY_ID", "SWIFT_CODE", "USERNAME", "USER_ID", "VEHICLE_ID"
    ]

    # Fix for tags like NAME> or /NAME>
    pattern = re.compile(r'(?<!<)(/?(?:' + "|".join(tag_names) + r'))>', re.IGNORECASE)
    fixed = pattern.sub(r'<\1>', text)

    return fixed


###Rephrasing Model (Old)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sentence_transformers import SentenceTransformer, util
import torch, re

# Lazy global objects
rephrase_pipe = None
similarity_model = None

def load_models():
    global rephrase_pipe, similarity_model
    if rephrase_pipe and similarity_model:
        return rephrase_pipe, similarity_model

    print("🔄 Loading Mistral-7B-Instruct and MiniLM for semantic rephrasing...")
    model_name = "mistralai/Mistral-7B-Instruct-v0.3"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )
    rephrase_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=150,
        temperature=0.3,
        do_sample=True
    )
    similarity_model = SentenceTransformer("all-MiniLM-L6-v2")
    return rephrase_pipe, similarity_model


def build_grammatical_rephrase_prompt(user_input):
    """
    Builds a prompt for purely grammatical rephrasing.
    Keeps all factual, numeric, and personal details untouched.
    """
    return (
        "You are a precise language corrector. "
        "Your task is to rewrite the following text with perfect grammar, punctuation, and clarity. "
        "Do NOT remove, replace, or generalize any information such as names, numbers, or account details. "
        "Keep the tone and meaning identical.\n\n"
        f"Text:\n{user_input}\n\n"
        "Grammatically improved version:"
    )


def rephrase_safe(user_input):
    pipe, _ = load_models()
    prompt = build_grammatical_rephrase_prompt(user_input)
    raw = pipe(prompt)[0]["generated_text"]
    result = raw.split("Grammatically improved version:")[-1].strip()
    result = re.sub(r"\s+([.,!?])", r"\1", result)
    return result

sanitized = "I am so angry! I've been on hold for 20 minutes. My name is , my phone is 9876543210, and my account is 887766. I just want to know my balance."
rephrased = rephrase_safe(sanitized)

🔄 Loading Mistral-7B-Instruct and MiniLM for semantic rephrasing...


KeyboardInterrupt: 

###Rephrase Test2

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

model.save_pretrained("/content/mistral_rephrase_model")
tokenizer.save_pretrained("/content/mistral_rephrase_model")


In [ ]:
!zip -r /content/mistral_rephrase_model.zip /content/mistral_rephrase_model

#NEW

###Train

In [ ]:
# ===========================================
# Imports
# ===========================================
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Trainer,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
import torch
import re
import json

# ===========================================
# Preprocess dataset (replace tags)
# ===========================================
# Define all tag types
tag_list = [
    "ACCOUNT_NUMBER", "ADDRESS", "APPLICATION_ID", "BRANCH_CODE", "CARD_NUMBER",
    "CHEQUE_NUMBER", "CUSTOMER_ID", "DATE_RANGE", "DOB", "DOCUMENT_ID", "EMAIL",
    "EMPLOYEE_ID", "FUND_ID", "ID_NUMBER", "IFSC_CODE", "NAME", "ORGANIZATION",
    "PHONE", "POLICY_ID", "SWIFT_CODE", "USERNAME", "USER_ID", "VEHICLE_ID"
]

# Function to replace tags with <SENSITIVE>
# def convert_to_simplified_tags(line):
#     for tag in tag_list:
#         line = re.sub(rf"<{tag}>", "<SENSITIVE>", line)
#         line = re.sub(rf"</{tag}>", "</SENSITIVE>", line)
#     return line

# # Read and rewrite JSONL dataset
# input_path = "new_cleaned.jsonl"
output_path = "new_cleaned.jsonl"

# with open(input_path, "r", encoding="utf-8") as infile, open(output_path, "w", encoding="utf-8") as outfile:
#     for line in infile:
#         data = json.loads(line)
#         data["original_query"] = convert_to_simplified_tags(data["original_query"])
#         data["sanitized_query"] = convert_to_simplified_tags(data["sanitized_query"])
#         outfile.write(json.dumps(data) + "\n")

# print(f"✅ Simplified dataset saved to {output_path}")

# ===========================================
# Load dataset
# ===========================================
dataset = load_dataset("json", data_files=output_path)
dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)
train_ds = dataset["train"]
val_ds = dataset["test"]

# ===========================================
# Model & Tokenizer
# ===========================================
model_name = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# right after tokenizer = AutoTokenizer.from_pretrained(model_name)
tag_tokens = [f"<{t}>" for t in tag_list] + [f"</{t}>" for t in tag_list]
tokenizer.add_tokens(tag_tokens)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))


# Add only the simplified tags
# special_tokens = ["<SENSITIVE>", "</SENSITIVE>"]
# tokenizer.add_tokens(special_tokens)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))

# ===========================================
# Preprocessing
# ===========================================
def preprocess(examples):
    inputs, targets = [], []
    for orig, sanitized in zip(examples["original_query"], examples["sanitized_query"]):
        if orig and sanitized:
            inputs.append("sanitize: " + str(orig))
            targets.append(str(sanitized))
    model_inputs = tokenizer(inputs, max_length=256, truncation=True)
    labels = tokenizer(targets, max_length=256, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

# ===========================================
# Data Collator
# ===========================================
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# ===========================================
# Training Arguments (Fixed)
# ===========================================
args = TrainingArguments(
    output_dir="./sanitizer_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_dir="./logs",
    logging_strategy="steps",
    logging_steps=50
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# ===========================================
# Trainer
# ===========================================
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# ===========================================
# Train the model
# ===========================================
trainer.train()

# ===========================================
# Save final model
# ===========================================
model.save_pretrained("./final_sanitizer_model")
tokenizer.save_pretrained("./final_sanitizer_model")

print("✅ Model saved to ./final_sanitizer_model")


Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

...

Map:   0%|          | 0/4042 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

/tmp/ipython-input-2464924516.py:111: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: saransh-naik14 (saransh-naik14-iiit-vadodara

Epoch,Training Loss,Validation Loss
1,0.145400,0.057049
2,0.074600,0.031829
3,0.053500,0.022631
4,0.038500,0.019554
5,0.038900,0.018056
6,0.031500,0.017177
7,0.028200,0.016462
8,0.029900,0.016051
9,0.026700,0.015852
10,0.029400,0.015906


✅ Model saved to ./final_sanitizer_model


In [ ]:
print("Predicted:", preds[0])
print("Actual:", batch["sanitized_query"][0])


Predicted: I need to set up a **standing instruction** to invest **5,000 Rs** into **Folio FUND_ID>9988/FUND_ID>** on the **15th of every month**.
Actual: I need to set up a **standing instruction** to invest **5,000 Rs** into **Folio <FUND_ID>9988</FUND_ID>** on the **15th of every month**.


###Test

In [ ]:
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from collections import defaultdict
import torch

# Load model & tokenizer
model_path = "./final_sanitizer_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda" if torch.cuda.is_available() else "cpu")

# Load dataset
output_path = "new_cleaned.jsonl"
dataset = load_dataset("json", data_files=output_path)["train"].train_test_split(test_size=0.1, seed=42)
val_ds = dataset["test"]

# Define all tag types
import re
from collections import defaultdict

tag_list = [
    "ACCOUNT_NUMBER","ADDRESS","APPLICATION_ID","BRANCH_CODE","CARD_NUMBER",
    "CHEQUE_NUMBER","CUSTOMER_ID","DATE_RANGE","DOB","DOCUMENT_ID","EMAIL",
    "EMPLOYEE_ID","FUND_ID","ID_NUMBER","IFSC_CODE","NAME","ORGANIZATION",
    "PHONE","POLICY_ID","SWIFT_CODE","USERNAME","USER_ID","VEHICLE_ID"
]

def repair_tags(text: str) -> str:
    """Fix common malformed tag patterns like TAG>...</TAG> and /TAG> without '<'."""
    for t in tag_list:
        # open tag missing '<'  -> TAG>  -> <TAG>
        text = re.sub(fr"(?<!<){t}>", f"<{t}>", text)
        # close tag missing '<' -> /TAG> -> </TAG>
        text = re.sub(fr"(?<!<)/{t}>", f"</{t}>", text)
    return text

def extract_tags(text: str):
    """Extract tags after repairing; accepts both <TAG> and </TAG>."""
    text = repair_tags(text)
    matches = re.findall(r"</?(\w+)>", text)
    return {m for m in matches if m in tag_list}

# ---- in your evaluation loop, wrap preds before counting ----
# pred = repair_tags(pred)   # if you also want to save/inspect the fixed string
# then use extract_tags(pred) and extract_tags(true)

# Generate predictions
def generate_predictions(batch):
    inputs = tokenizer(
        ["sanitize: " + x for x in batch["original_query"]],
        max_length=256,
        truncation=True,
        padding="max_length",
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=256,
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# Metrics
tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)

# Evaluate tag predictions
for i in range(0, len(val_ds), 8):  # batch size 8
    batch = val_ds[i:i+8]
    preds = generate_predictions(batch)

    for pred, true in zip(preds, batch["sanitized_query"]):
        pred_tags = extract_tags(pred)
        true_tags = extract_tags(true)

        for tag in tag_list:
            if tag in pred_tags and tag in true_tags:
                tp[tag] += 1
            elif tag in pred_tags and tag not in true_tags:
                fp[tag] += 1
            elif tag not in pred_tags and tag in true_tags:
                fn[tag] += 1

# Compute and print results
def compute_f1(tp, fp, fn):
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1

print(f"{'Tag':<20} {'Prec':<10} {'Recall':<10} {'F1':<10}")
for tag in tag_list:
    if tp[tag] + fp[tag] + fn[tag] == 0:
        continue  # Skip tags not present in val
    p, r, f1 = compute_f1(tp[tag], fp[tag], fn[tag])
    print(f"{tag:<20} {p:<10.2f} {r:<10.2f} {f1:<10.2f}")




Tag                  Prec       Recall     F1        
ACCOUNT_NUMBER       0.98       0.99       0.99      
ADDRESS              0.55       0.75       0.63      
APPLICATION_ID       0.77       0.92       0.84      
CARD_NUMBER          1.00       1.00       1.00      
CHEQUE_NUMBER        0.00       0.00       0.00      
CUSTOMER_ID          0.40       0.67       0.50      
DOB                  0.00       0.00       0.00      
DOCUMENT_ID          1.00       0.50       0.67      
EMAIL                0.00       0.00       0.00      
EMPLOYEE_ID          1.00       0.94       0.97      
FUND_ID              0.71       0.71       0.71      
ID_NUMBER            0.93       0.75       0.83      
NAME                 1.00       1.00       1.00      
ORGANIZATION         0.97       0.94       0.95      
PHONE                1.00       1.00       1.00      
POLICY_ID            1.00       0.50       0.67      
VEHICLE_ID           0.00       0.00       0.00      


In [ ]:
correct_examples = 0
total_examples = 0

for i in range(0, len(val_ds), 8):  # batch size = 8
    batch = val_ds[i:i+8]
    preds = generate_predictions(batch)

    for pred, true in zip(preds, batch["sanitized_query"]):
        pred_tags = extract_tags(pred)
        true_tags = extract_tags(true)

        if pred_tags == true_tags:  # exact tag match
            correct_examples += 1
        total_examples += 1

accuracy = correct_examples / total_examples
print(f"\nExact Tag-Match Accuracy: {accuracy:.2%}")



Exact Tag-Match Accuracy: 89.33%


In [ ]:
from sklearn.metrics import multilabel_confusion_matrix
import numpy as np

# List of evaluated tags (remove zero-F1 tags if needed)
eval_tags = [
    "ACCOUNT_NUMBER", "ADDRESS", "APPLICATION_ID", "CARD_NUMBER",
    "CUSTOMER_ID", "DOCUMENT_ID", "EMPLOYEE_ID", "FUND_ID",
    "ID_NUMBER", "NAME", "ORGANIZATION", "PHONE", "POLICY_ID"
]

# Initialize lists for true and predicted binary vectors
true_binary = []
pred_binary = []

# Iterate again over validation examples
for i in range(0, len(val_ds), 8):  # batch size = 8
    batch = val_ds[i:i+8]
    preds = generate_predictions(batch)

    for pred, true in zip(preds, batch["sanitized_query"]):
        pred_tags = extract_tags(pred)
        true_tags = extract_tags(true)

        true_binary.append([1 if tag in true_tags else 0 for tag in eval_tags])
        pred_binary.append([1 if tag in pred_tags else 0 for tag in eval_tags])

# Convert to NumPy arrays
true_binary = np.array(true_binary)
pred_binary = np.array(pred_binary)

# Compute confusion matrices
conf_matrices = multilabel_confusion_matrix(true_binary, pred_binary)

# Display confusion matrices
for tag, matrix in zip(eval_tags, conf_matrices):
    print(f"\nConfusion Matrix for tag '{tag}':")
    print(matrix)



Confusion Matrix for tag 'ACCOUNT_NUMBER':
[[186   4]
 [  2 258]]

Confusion Matrix for tag 'ADDRESS':
[[424  10]
 [  4  12]]

Confusion Matrix for tag 'APPLICATION_ID':
[[386  14]
 [  4  46]]

Confusion Matrix for tag 'CARD_NUMBER':
[[407   0]
 [  0  43]]

Confusion Matrix for tag 'CUSTOMER_ID':
[[444   3]
 [  1   2]]

Confusion Matrix for tag 'DOCUMENT_ID':
[[448   0]
 [  1   1]]

Confusion Matrix for tag 'EMPLOYEE_ID':
[[433   0]
 [  1  16]]

Confusion Matrix for tag 'FUND_ID':
[[441   2]
 [  2   5]]

Confusion Matrix for tag 'ID_NUMBER':
[[395   3]
 [ 13  39]]

Confusion Matrix for tag 'NAME':
[[383   0]
 [  0  67]]

Confusion Matrix for tag 'ORGANIZATION':
[[417   1]
 [  2  30]]

Confusion Matrix for tag 'PHONE':
[[444   0]
 [  0   6]]

Confusion Matrix for tag 'POLICY_ID':
[[448   0]
 [  1   1]]


In [ ]:
sample = "This account <ACCOUNT_NUMBER>121212</ACCOUNT_NUMBER> belongs to <ORGANIZATION>Import Co</ORGANIZATION>."
print(extract_tags(sample))


{'ORGANIZATION', 'ACCOUNT_NUMBER'}
